In [ ]:
from pathlib import Path
import polars as pl

data_dir = Path().absolute() / ".." / "data" / "parsed" / "full_takeout"

df = pl.read_parquet(data_dir / "clxenc3fw0007gzz3pdz6enfe.snappy")

In [ ]:
daily_dfs = df.with_columns(
    date=pl.col("timestamp").dt.date()
).partition_by("date", as_dict=True, include_key=False)


In [ ]:
chunk_size = 10
chunks={}

for date, day_df in daily_dfs.items():
    chunks[date] = []
    for slice in day_df.iter_slices(chunk_size):
        chunks[date].append(slice.select("hour", "title"))

In [ ]:
def df_to_text(df):
    text_output = ""
    for row in df.iter_rows():
        hour, title = row
        text_output += f"{hour}: {title}\n"
    return text_output

In [ ]:
import datetime
d = chunks[datetime.date(2023, 8, 17)][0]

print(df_to_text(d))